# 02 - 行为日志模拟 & 指标计算

目标：
1. 基于真实订单数据模拟访问/浏览/加购行为日志
2. 计算 DAU、漏斗、留存、复购、RFM 等核心指标
3. 输出看板所需的 JSON 数据

In [ ]:
import pandas as pd
import numpy as np
import json
import os

PROCESSED = '../data/processed'
SIMULATED = '../data/simulated'
DASHBOARD_DATA = '../dashboard/public/data'
os.makedirs(SIMULATED, exist_ok=True)
os.makedirs(DASHBOARD_DATA, exist_ok=True)

np.random.seed(42)

## 1. 加载清洗后的数据

In [ ]:
order_wide = pd.read_csv(f'{PROCESSED}/order_wide.csv', parse_dates=['order_purchase_timestamp', 'order_approved_at'])
first_orders = pd.read_csv(f'{PROCESSED}/customer_first_orders.csv', parse_dates=['first_order_date'])

print(f'订单宽表: {len(order_wide):,} 行')
print(f'客户数: {order_wide.customer_unique_id.nunique():,}')
print(f'时间范围: {order_wide.order_purchase_timestamp.min()} ~ {order_wide.order_purchase_timestamp.max()}')

## 2. 模拟行为日志

转化率参数设定（基于行业基准）：
- 访问 → 浏览：60%
- 浏览 → 加购：15%
- 加购 → 下单：40%
- 下单 → 支付：95%

每个真实下单用户反向推导其行为链路。

In [ ]:
# 转化率参数
CONV_VIEW = 0.60    # 访问 → 浏览
CONV_CART = 0.15    # 浏览 → 加购
CONV_ORDER = 0.40   # 加购 → 下单
CONV_PAY = 0.95     # 下单 → 支付

# 获取有效下单客户
order_customers = order_wide[['customer_unique_id', 'order_id', 'order_purchase_timestamp', 'product_id']].drop_duplicates()

# 反向推导：每个下单用户 → 加购 → 浏览 → 访问
# 1 下单用户对应 CONV_ORDER 个加购用户
# 1 加购用户对应 CONV_CART 个浏览用户
# 1 浏览用户对应 CONV_VIEW 个访问用户

n_orders = len(order_customers)
n_cart = int(n_orders / CONV_ORDER)       # 加购用户数
n_view = int(n_cart / CONV_CART)          # 浏览用户数
n_visit = int(n_view / CONV_VIEW)         # 访问用户数

print(f'模拟规模:')
print(f'  访问: {n_visit:,}')
print(f'  浏览: {n_view:,}')
print(f'  加购: {n_cart:,}')
print(f'  下单: {n_orders:,}')

In [ ]:
# 生成行为日志
# 策略：以真实订单时间为锚点，向前推算各环节时间

def random_time_before(anchor_ts, min_hours, max_hours):
    """在锚点时间前随机生成一个时间"""
    offset = pd.Timedelta(hours=np.random.uniform(min_hours, max_hours))
    return anchor_ts - offset

devices = ['mobile', 'desktop']
sources = ['organic', 'paid', 'social', 'direct']
device_weights = [0.62, 0.38]
source_weights = [0.45, 0.25, 0.18, 0.12]

logs = []
log_id = 0

# 取前 50000 个订单做模拟（控制数据量）
sample_orders = order_customers.head(50000)

for idx, row in sample_orders.iterrows():
    cust_id = row['customer_unique_id']
    order_ts = row['order_purchase_timestamp']
    prod_id = row['product_id']
    session_id = f's_{log_id:08d}'
    device = np.random.choice(devices, p=device_weights)
    source = np.random.choice(sources, p=source_weights)

    # 下单事件
    logs.append([log_id, cust_id, session_id, 'order', order_ts, prod_id, device, source])
    log_id += 1

    # 支付事件（95% 概率）
    if np.random.random() < CONV_PAY:
        pay_ts = order_ts + pd.Timedelta(minutes=np.random.uniform(1, 30))
        logs.append([log_id, cust_id, session_id, 'payment', pay_ts, prod_id, device, source])
        log_id += 1

    # 加购事件（在下单前 5-120 分钟）
    cart_ts = random_time_before(order_ts, 0.08, 2)
    logs.append([log_id, cust_id, session_id, 'add_to_cart', cart_ts, prod_id, device, source])
    log_id += 1

# 补充未转化的浏览和访问（随机时间分布在整个时间范围内）
time_min = order_wide.order_purchase_timestamp.min()
time_max = order_wide.order_purchase_timestamp.max()
time_range_days = (time_max - time_min).days

# 浏览事件（未加购的）
extra_views = n_view - n_orders
for i in range(min(extra_views, 80000)):
    day_offset = np.random.uniform(0, time_range_days)
    ts = time_min + pd.Timedelta(days=day_offset, hours=np.random.uniform(6, 23))
    cust_id = f'v_{i:08d}'
    session_id = f'sv_{i:08d}'
    device = np.random.choice(devices, p=device_weights)
    source = np.random.choice(sources, p=source_weights)
    prod_id = np.random.choice(order_wide.product_id.dropna().unique())
    logs.append([log_id, cust_id, session_id, 'view', ts, prod_id, device, source])
    log_id += 1

# 访问事件（未浏览的）
extra_visits = n_visit - n_view
for i in range(min(extra_visits, 120000)):
    day_offset = np.random.uniform(0, time_range_days)\n    ts = time_min + pd.Timedelta(days=day_offset, hours=np.random.uniform(6, 23))
    cust_id = f'vv_{i:08d}'
    session_id = f'svv_{i:08d}'
    device = np.random.choice(devices, p=device_weights)
    source = np.random.choice(sources, p=source_weights)
    logs.append([log_id, cust_id, session_id, 'visit', ts, None, device, source])
    log_id += 1

behavior_log = pd.DataFrame(logs, columns=['log_id', 'customer_id', 'session_id', 'event_type', 'event_timestamp', 'product_id', 'device', 'source'])
behavior_log = behavior_log.sort_values('event_timestamp').reset_index(drop=True)

print(f'\n行为日志总条数: {len(behavior_log):,}')
print(f'事件类型分布:')
print(behavior_log['event_type'].value_counts())

In [ ]:
# 保存行为日志
behavior_log.to_csv(f'{SIMULATED}/behavior_log.csv', index=False)
print(f'已保存: simulated/behavior_log.csv ({len(behavior_log):,} 行)')

## 3. 计算核心指标

In [ ]:
# --- KPI 总览 ---
total_gmv = order_wide['payment_value'].sum()
total_orders = order_wide['order_id'].nunique()
aov = total_gmv / total_orders
unique_customers = order_wide['customer_unique_id'].nunique()

# 转化率 = 支付用户数 / 访问用户数
n_pay = len(behavior_log[behavior_log['event_type'] == 'payment'])
n_visit_total = len(behavior_log[behavior_log['event_type'] == 'visit'])
conversion_rate = n_pay / n_visit_total if n_visit_total > 0 else 0

# DAU
behavior_log['event_date'] = pd.to_datetime(behavior_log['event_timestamp']).dt.date
dau_daily = behavior_log.groupby('event_date')['customer_id'].nunique().reset_index()
dau_daily.columns = ['date', 'dau']
avg_dau = int(dau_daily['dau'].mean())

# 30日留存（简化：以最后30天有活跃的用户占比）
last_date = pd.to_datetime(behavior_log['event_timestamp']).max()
last_30d = behavior_log[pd.to_datetime(behavior_log['event_timestamp']) >= last_date - pd.Timedelta(days=30)]
retention_30d = last_30d['customer_id'].nunique() / behavior_log['customer_id'].nunique()

kpi = {
    'totalGMV': f'R$ {total_gmv:,.0f}'.replace(',', ','),
    'totalOrders': f'{total_orders:,}',
    'avgOrderValue': f'R$ {aov:,.2f}',
    'conversionRate': f'{conversion_rate*100:.1f}%',
    'dau': f'{avg_dau:,}',
    'retention30d': f'{retention_30d*100:.1f}%',
}
print('KPI:')
for k, v in kpi.items():
    print(f'  {k}: {v}')

In [ ]:
# --- 月度 GMV ---
order_wide['month'] = order_wide['order_purchase_timestamp'].dt.to_period('M').astype(str)
monthly_gmv = (
    order_wide.groupby('month')
    .agg(gmv=('payment_value', 'sum'), orders=('order_id', 'nunique'))
    .reset_index()
)
monthly_gmv['gmv'] = monthly_gmv['gmv'].round(0).astype(int)
print(f'月度GMV: {len(monthly_gmv)} 个月')
print(monthly_gmv.head())

In [ ]:
# --- 月度 DAU ---
behavior_log['month'] = pd.to_datetime(behavior_log['event_timestamp']).dt.to_period('M').astype(str)
monthly_dau = (
    behavior_log.groupby('month')
    .apply(lambda g: pd.Series({
        'avgDau': int(g.groupby('event_date')['customer_id'].nunique().mean()),
        'peakDau': int(g.groupby('event_date')['customer_id'].nunique().max())
    }))
    .reset_index()
)
print(f'月度DAU: {len(monthly_dau)} 个月')

In [ ]:
# --- 转化漏斗 ---
funnel = (
    behavior_log[behavior_log['event_type'].isin(['visit', 'view', 'add_to_cart', 'order', 'payment'])]
    .groupby('event_type')['customer_id'].nunique()
    .reindex(['visit', 'view', 'add_to_cart', 'order', 'payment'])
    .reset_index()
)
funnel.columns = ['step', 'users']
total_visit = funnel.iloc[0]['users']
funnel['rate'] = (funnel['users'] / total_visit * 100).round(1).astype(str) + '%'

# 中文映射
step_map = {'visit': '访问', 'view': '浏览商品', 'add_to_cart': '加入购物车', 'order': '提交订单', 'payment': '完成支付'}
funnel['step'] = funnel['step'].map(step_map)
print('转化漏斗:')
print(funnel)

In [ ]:
# --- 品类 GMV ---
cat_col = 'product_category_name_english'
if cat_col not in order_wide.columns:
    cat_col = 'product_category_name'

category_gmv = (
    order_wide.groupby(cat_col)
    .agg(gmv=('payment_value', 'sum'), orders=('order_id', 'nunique'))
    .sort_values('gmv', ascending=False)
    .head(10)
    .reset_index()
)
category_gmv.columns = ['category', 'gmv', 'orders']
category_gmv['gmv'] = category_gmv['gmv'].round(0).astype(int)
category_gmv['category'] = category_gmv['category'].fillna('Unknown')
print('品类GMV Top10:')
print(category_gmv)

In [ ]:
# --- 地区 GMV ---
region_gmv = (
    order_wide.groupby('customer_state')
    .agg(gmv=('payment_value', 'sum'), orders=('order_id', 'nunique'), customers=('customer_unique_id', 'nunique'))
    .sort_values('gmv', ascending=False)
    .head(10)
    .reset_index()
)
region_gmv.columns = ['state', 'gmv', 'orders', 'customers']
region_gmv['gmv'] = region_gmv['gmv'].round(0).astype(int)
print('地区GMV Top10:')
print(region_gmv)

In [ ]:
# --- 转化率趋势 ---
# 按月计算各环节转化率
bl = behavior_log.copy()
bl['event_date'] = pd.to_datetime(bl['event_timestamp']).dt.date

monthly_funnel = (
    bl[bl['event_type'].isin(['visit', 'view', 'add_to_cart', 'order', 'payment'])]
    .groupby(['month', 'event_type'])['customer_id'].nunique()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ['visit', 'view', 'add_to_cart', 'order', 'payment']:
    if col not in monthly_funnel.columns:
        monthly_funnel[col] = 0

monthly_funnel['visitToView'] = (monthly_funnel['view'] / monthly_funnel['visit'].clip(lower=1) * 100).round(1)
monthly_funnel['viewToCart'] = (monthly_funnel['add_to_cart'] / monthly_funnel['view'].clip(lower=1) * 100).round(1)
monthly_funnel['cartToOrder'] = (monthly_funnel['order'] / monthly_funnel['add_to_cart'].clip(lower=1) * 100).round(1)
monthly_funnel['orderToPay'] = (monthly_funnel['payment'] / monthly_funnel['order'].clip(lower=1) * 100).round(1)

conversion_trend = monthly_funnel[['month', 'visitToView', 'viewToCart', 'cartToOrder', 'orderToPay']]
print('转化率趋势:')
print(conversion_trend.head())

In [ ]:
# --- 留存 Cohort ---
# 以用户首次下单月为 cohort，计算后续月份的活跃留存
first_orders['cohort'] = first_orders['first_order_date'].dt.to_period('M').astype(str)

# 用户活跃月份
user_active = (
    bl[bl['event_type'].isin(['order', 'payment'])]
    .copy()
)
user_active['active_month'] = pd.to_datetime(user_active['event_timestamp']).dt.to_period('M').astype(str)
user_active = user_active[['customer_id', 'active_month']].drop_duplicates()

# 只取真实客户（非模拟ID）
real_customers = set(first_orders['customer_unique_id'])
user_active = user_active[user_active['customer_id'].isin(real_customers)]

# 合并 cohort
user_active = user_active.merge(
    first_orders[['customer_unique_id', 'cohort']],
    left_on='customer_id', right_on='customer_unique_id', how='left'
)

# 计算月份差
user_active['cohort_dt'] = pd.to_datetime(user_active['cohort'] + '-01')
user_active['active_dt'] = pd.to_datetime(user_active['active_month'] + '-01')
user_active['month_offset'] = ((user_active['active_dt'].dt.year - user_active['cohort_dt'].dt.year) * 12 +
                                (user_active['active_dt'].dt.month - user_active['cohort_dt'].dt.month))

# 留存表
cohort_size = user_active[user_active['month_offset'] == 0].groupby('cohort')['customer_id'].nunique()
retention = (
    user_active.groupby(['cohort', 'month_offset'])['customer_id'].nunique()
    .unstack(fill_value=0)
)

# 转为百分比
retention_pct = retention.div(cohort_size, axis=0) * 100

# 取前 6 个 cohort，保留 0-5 月
retention_pct = retention_pct.head(6)
retention_pct = retention_pct[[c for c in range(6) if c in retention_pct.columns]]
retention_pct = retention_pct.round(1)

print('留存Cohort (%):')
print(retention_pct)

In [ ]:
# --- RFM 分层 ---
reference_date = order_wide['order_purchase_timestamp'].max()

rfm = (
    order_wide.groupby('customer_unique_id')
    .agg(
        recency=('order_purchase_timestamp', lambda x: (reference_date - x.max()).days),
        frequency=('order_id', 'nunique'),
        monetary=('payment_value', 'sum')
    )
    .reset_index()
)

# 三分位打分
r33, r67 = rfm['recency'].quantile([0.33, 0.67])
f33, f67 = rfm['frequency'].quantile([0.33, 0.67])
m33, m67 = rfm['monetary'].quantile([0.33, 0.67])

rfm['r_score'] = np.where(rfm['recency'] <= r33, 3, np.where(rfm['recency'] <= r67, 2, 1))
rfm['f_score'] = np.where(rfm['frequency'] >= f67, 3, np.where(rfm['frequency'] >= f33, 2, 1))
rfm['m_score'] = np.where(rfm['monetary'] >= m67, 3, np.where(rfm['monetary'] >= m33, 2, 1))

# 分层
def assign_segment(row):
    r, f, m = row['r_score'], row['f_score'], row['m_score']
    if r == 3 and f >= 2 and m >= 2:
        return '高价值用户'
    elif r == 3 and (f == 1 or m == 1):
        return '潜力用户'
    elif r == 1 and (f >= 2 or m >= 2):
        return '沉睡用户'
    elif r == 1 and f == 1 and m == 1:
        return '流失风险'
    else:
        return '一般用户'

rfm['segment'] = rfm.apply(assign_segment, axis=1)

rfm_summary = (
    rfm.groupby('segment')
    .agg(
        count=('customer_unique_id', 'count'),
        avgRecency=('recency', 'mean'),
        avgFrequency=('frequency', 'mean'),
        avgMonetary=('monetary', 'mean')
    )
    .reset_index()
)
rfm_summary['pct'] = (rfm_summary['count'] / rfm_summary['count'].sum() * 100).round(1)
rfm_summary['avgRecency'] = rfm_summary['avgRecency'].round(0).astype(int)
rfm_summary['avgFrequency'] = rfm_summary['avgFrequency'].round(1)
rfm_summary['avgMonetary'] = rfm_summary['avgMonetary'].round(1)
rfm_summary.columns = ['segment', 'count', 'avgRecency', 'avgFrequency', 'avgMonetary', 'pct']

print('RFM 分层:')
print(rfm_summary)

## 4. 输出看板 JSON

In [ ]:
def to_json_records(df):
    return json.loads(df.to_json(orient='records', force_ascii=False))

# KPI
with open(f'{DASHBOARD_DATA}/kpi.json', 'w', encoding='utf-8') as f:
    json.dump(kpi, f, ensure_ascii=False, indent=2)

# 月度GMV
with open(f'{DASHBOARD_DATA}/monthly_gmv.json', 'w', encoding='utf-8') as f:
    json.dump(to_json_records(monthly_gmv), f, ensure_ascii=False)

# 月度DAU
with open(f'{DASHBOARD_DATA}/monthly_dau.json', 'w', encoding='utf-8') as f:
    json.dump(to_json_records(monthly_dau), f, ensure_ascii=False)

# 漏斗
with open(f'{DASHBOARD_DATA}/funnel.json', 'w', encoding='utf-8') as f:
    json.dump(to_json_records(funnel), f, ensure_ascii=False)

# 品类GMV
with open(f'{DASHBOARD_DATA}/category_gmv.json', 'w', encoding='utf-8') as f:
    json.dump(to_json_records(category_gmv), f, ensure_ascii=False)

# 地区GMV
with open(f'{DASHBOARD_DATA}/region_gmv.json', 'w', encoding='utf-8') as f:
    json.dump(to_json_records(region_gmv), f, ensure_ascii=False)

# 转化率趋势
with open(f'{DASHBOARD_DATA}/conversion_trend.json', 'w', encoding='utf-8') as f:
    json.dump(to_json_records(conversion_trend), f, ensure_ascii=False)

# 留存Cohort
retention_data = []
for cohort, row in retention_pct.iterrows():
    item = {'cohort': cohort}
    for offset in range(6):
        key = f'month{offset}'
        item[key] = row.get(offset, None)
        if pd.notna(item[key]):
            item[key] = round(float(item[key]), 1)
        else:
            item[key] = None
    retention_data.append(item)

with open(f'{DASHBOARD_DATA}/retention_cohort.json', 'w', encoding='utf-8') as f:
    json.dump(retention_data, f, ensure_ascii=False)

# RFM
with open(f'{DASHBOARD_DATA}/rfm_segments.json', 'w', encoding='utf-8') as f:
    json.dump(to_json_records(rfm_summary), f, ensure_ascii=False)

print('已输出 JSON:')
for f in sorted(os.listdir(DASHBOARD_DATA)):
    size = os.path.getsize(f'{DASHBOARD_DATA}/{f}')
    print(f'  {f}: {size:,} bytes')